# Homework 4: Misleading Visualization Pair
**False claim pushed by Figure 1:** marijuana is overwhelmingly what CBP seizes; meth is minor.

**True fact shown by Figure 2:** by weight, CBP seizes nearly as much meth as marijuana.

Data: CBP Drug Seizure Statistics, `drugs_seized.csv`.

In [ ]:
# Data 227 HW4 - Misleading / honest visualization pair
# Data: U.S. Customs and Border Protection, Drug Seizure Statistics (CBP Public Data Portal)
#       https://www.cbp.gov/newsroom/stats/drug-seizure-statistics  -> drugs_seized.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------------- data prep
raw = pd.read_csv("drugs_seized.csv")

# "Other Drugs**" is CBP's catch-all category, not a single drug, so it is
# left out of BOTH charts (stated in both captions).
raw = raw[raw["Drug Type"] != "Other Drugs**"]
raw["Drug Type"] = raw["Drug Type"].replace({"Lsd": "LSD"})

by_year = (raw.groupby(["FY", "Drug Type"])[["Count of Event", "Sum Qty (lbs)"]]
              .sum().reset_index())

events = by_year.pivot(index="Drug Type", columns="FY", values="Count of Event")
pounds = by_year.pivot(index="Drug Type", columns="FY", values="Sum Qty (lbs)")
years = list(events.columns)

## Figure 1: Misleading

In [ ]:
# ======================================================= 1. MISLEADING CHART
# Techniques: event counts presented as "drug seizures"; years collapsed into
# one total; one loud color vs. gray; assertive title; share-of-total callout;
# only the marijuana bar is labeled; per-day framing; official source line.
ev_total = events.sum(axis=1).sort_values(ascending=False)
share = ev_total / ev_total.sum() * 100
DAYS = 1096                      # FY2023-FY2025: Oct 1, 2022 - Sep 30, 2025
per_day = ev_total["Marijuana"] / DAYS

fig, ax = plt.subplots(figsize=(11, 6.8))
colors = ["#1a9e3f" if d == "Marijuana" else "#c4c4c4" for d in ev_total.index]
bars = ax.bar(ev_total.index, ev_total.values, color=colors, edgecolor="white")

# label ONLY marijuana; the gray bars get no numbers to compare against
ax.text(bars[0].get_x() + bars[0].get_width() / 2, ev_total["Marijuana"] + 900,
        f"{ev_total['Marijuana']:,.0f}", ha="center", fontsize=13,
        fontweight="bold", color="#1a9e3f")

ax.annotate(f"Marijuana: {share['Marijuana']:.0f}% of all seizures",
            xy=(0.45, ev_total["Marijuana"] * 0.85), xytext=(1.5, ev_total["Marijuana"] * 0.80),
            fontsize=14, fontweight="bold", color="#1a9e3f",
            arrowprops=dict(arrowstyle="->", color="#1a9e3f", lw=2))

fig.suptitle("Marijuana Dominates Drug Seizures at U.S. Borders",
             fontsize=18, fontweight="bold", x=0.085, ha="left", y=0.955)
ax.set_title(f"CBP made about {per_day:.0f} marijuana seizures every day, FY2023–FY2025",
             fontsize=12, color="#444444", loc="left")
ax.set_ylabel("Drug seizures, FY2023–FY2025 combined", fontsize=12)
ax.set_xlabel("Drug type", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
plt.xticks(rotation=30, ha="right")
fig.text(0.085, 0.015, "Source: U.S. Customs and Border Protection, Drug Seizure Statistics",
         fontsize=9, color="#666666")
fig.subplots_adjust(left=0.085, right=0.98, top=0.86, bottom=0.2)
plt.savefig("misleading.png", dpi=200)
plt.show()

## Figure 2: Honest

In [ ]:
# ========================================================== 2. HONEST CHART
# Same data, same years, same drugs: event counts and pounds side by side,
# year-by-year, neutral colors, explicit units, plain descriptive title.
order = pounds.sum(axis=1).sort_values().index      # smallest at bottom
y = np.arange(len(order))
h = 0.8 / len(years)
year_colors = ["#9ecae1", "#4292c6", "#08519c"]

fig, (axL, axR) = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
for i, fy in enumerate(years):
    off = (i - 1) * h
    axL.barh(y + off, events.loc[order, fy], height=h, color=year_colors[i], label=f"FY{fy}")
    axR.barh(y + off, pounds.loc[order, fy], height=h, color=year_colors[i], label=f"FY{fy}")

axL.set_yticks(y)
axL.set_yticklabels(order, fontsize=11)
axL.set_title("Number of seizure events", fontsize=13, fontweight="bold")
axL.set_xlabel("Seizure events (count of separate seizures)")
axR.set_title("Total weight seized", fontsize=13, fontweight="bold")
axR.set_xlabel("Pounds (lbs)")
for a in (axL, axR):
    a.spines[["top", "right"]].set_visible(False)
    a.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    a.grid(axis="x", alpha=0.3)

# label each year's value for the two drugs the comparison hinges on
for d in ["Marijuana", "Methamphetamine"]:
    j = list(order).index(d)
    for i, fy in enumerate(years):
        off = (i - 1) * h
        axL.text(events.loc[d, fy] + 300, j + off, f"{events.loc[d, fy]:,.0f}",
                 va="center", fontsize=8)
        axR.text(pounds.loc[d, fy] + 2000, j + off, f"{pounds.loc[d, fy]:,.0f}",
                 va="center", fontsize=8)

axR.legend(title="Fiscal year", loc="lower right", frameon=False)
fig.suptitle("By weight, CBP seizes nearly as much methamphetamine as marijuana,\n"
             "even though marijuana seizures happen about 7 times as often",
             fontsize=15, fontweight="bold", x=0.02, ha="left")
plt.tight_layout()
plt.savefig("honest.png", dpi=200)
plt.show()

# numbers used in the captions
print(ev_total, share.round(1), pounds.round(0), sep="\n\n")